# 03 — Dataset Merge (Synthetic Linkage)
## SBDR Phase A, Step A3
Merge all 5 processed datasets into one multi-modal record per customer.
UCI Credit Card (30K customers) is our anchor — everything maps onto these IDs.

In [1]:
import pandas as pd
import numpy as np

# Load all processed datasets
uci = pd.read_csv("../data/processed/uci_credit_processed.csv")
lc = pd.read_csv("../data/processed/lending_club_processed.csv")
sparkov = pd.read_csv("../data/processed/sparkov_processed.csv")
phrasebank = pd.read_csv("../data/processed/financial_phrasebank_processed.csv")
mental = pd.read_csv("../data/processed/mental_health_processed.csv")

print("=== All Datasets Loaded ===")
for name, df in [("UCI", uci), ("Lending Club", lc), ("Sparkov", sparkov), 
                  ("PhraseBank", phrasebank), ("Mental Health", mental)]:
    print(f"{name:15s} → {df.shape[0]:>10,} rows × {df.shape[1]} cols")

=== All Datasets Loaded ===
UCI             →     30,000 rows × 32 cols
Lending Club    →  2,257,919 rows × 19 cols
Sparkov         →  1,852,394 rows × 9 cols
PhraseBank      →      2,264 rows × 2 cols
Mental Health   →     38,175 rows × 2 cols


## Step 1: Create Customer IDs & Distress Levels
Each UCI customer gets a unique ID. Their payment history (PAY_0 to PAY_6) 
determines a "distress_level" which controls how we assign sentiment text and 
Lending Club profiles to them.

In [2]:
# Assign unique customer IDs
uci['customer_id'] = [f"SBDR_{i:05d}" for i in range(len(uci))]

# Compute distress level from payment history
# PAY columns: -2=no usage, -1=paid on time, 0=revolving, 1+=months late
pay_cols = ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']

# Average payment delay across 6 months (higher = worse)
uci['avg_pay_delay'] = uci[pay_cols].mean(axis=1)

# Distress level: 4 bins matching our 4 SBDR tiers
uci['distress_level'] = pd.cut(
    uci['avg_pay_delay'],
    bins=[-np.inf, -0.5, 0.0, 1.0, np.inf],
    labels=['low', 'moderate', 'high', 'severe']
)

print("=== Distress Level Distribution ===")
print(uci['distress_level'].value_counts().sort_index())
print(f"\navg_pay_delay range: {uci['avg_pay_delay'].min():.2f} to {uci['avg_pay_delay'].max():.2f}")

=== Distress Level Distribution ===
distress_level
low         10253
moderate    12614
high         4500
severe       2633
Name: count, dtype: int64

avg_pay_delay range: -2.00 to 6.00


## Step 2: Enrich with Lending Club Profiles
We compute aggregate stats from the full 2.26M Lending Club loans, 
grouped by risk tier. Then map those stats onto UCI customers 
based on their distress level.

In [3]:
# Map Lending Club sbdr_tier to our distress levels
tier_to_distress = {
    'No Action': 'low',
    'Tier 1 - Reminder': 'low',
    'Tier 2 - Assistance': 'moderate',
    'Tier 3 - Restructure': 'high',
    'Tier 4 - Legal': 'severe'
}
lc['distress_level'] = lc['sbdr_tier'].map(tier_to_distress)

# Drop rows that didn't map (e.g. "Current" loans with no tier)
lc_mapped = lc.dropna(subset=['distress_level'])
print(f"Lending Club rows with distress mapping: {len(lc_mapped):,} / {len(lc):,}")

# Compute aggregate profiles per distress level
# These are the numeric columns we want stats for
lc_numeric = ['loan_amnt', 'funded_amnt', 'annual_inc', 'dti', 
              'revol_util', 'delinq_2yrs', 'inq_last_6mths', 
              'open_acc', 'pub_rec', 'total_acc', 'installment', 'int_rate']

lc_profiles = lc_mapped.groupby('distress_level')[lc_numeric].agg(['mean', 'std']).reset_index()

# Flatten column names: "loan_amnt_mean", "loan_amnt_std", etc.
lc_profiles.columns = ['distress_level'] + [f"lc_{col}_{stat}" for col, stat in lc_profiles.columns[1:]]

print(f"\nLC profile shape: {lc_profiles.shape}")
print(f"Distress levels in profiles: {lc_profiles['distress_level'].tolist()}")

Lending Club rows with distress mapping: 1,955,068 / 2,257,919

LC profile shape: (1, 25)
Distress levels in profiles: ['low']


In [4]:
print("=== Lending Club sbdr_tier distribution ===")
print(lc['sbdr_tier'].value_counts())

=== Lending Club sbdr_tier distribution ===
sbdr_tier
No Action    1955068
Tier 4        268599
Tier 3         21467
Tier 1          8436
Tier 2          4349
Name: count, dtype: int64


---
Problem — almost everything mapped to "low."

---

In [5]:
# Correct mapping with actual tier labels
tier_to_distress = {
    'No Action': 'low',
    'Tier 1': 'low',
    'Tier 2': 'moderate',
    'Tier 3': 'high',
    'Tier 4': 'severe'
}
lc['distress_level'] = lc['sbdr_tier'].map(tier_to_distress)

lc_mapped = lc.dropna(subset=['distress_level'])
print(f"Lending Club rows with distress mapping: {len(lc_mapped):,} / {len(lc):,}")
print(f"\n=== LC Distress Distribution ===")
print(lc_mapped['distress_level'].value_counts().sort_index())

# Recompute profiles
lc_numeric = ['loan_amnt', 'funded_amnt', 'annual_inc', 'dti', 
              'revol_util', 'delinq_2yrs', 'inq_last_6mths', 
              'open_acc', 'pub_rec', 'total_acc', 'installment', 'int_rate']

lc_profiles = lc_mapped.groupby('distress_level')[lc_numeric].agg(['mean', 'std']).reset_index()
lc_profiles.columns = ['distress_level'] + [f"lc_{col}_{stat}" for col, stat in lc_profiles.columns[1:]]

print(f"\nLC profile shape: {lc_profiles.shape}")
print(f"Distress levels in profiles: {lc_profiles['distress_level'].tolist()}")

Lending Club rows with distress mapping: 2,257,919 / 2,257,919

=== LC Distress Distribution ===
distress_level
high          21467
low         1963504
moderate       4349
severe       268599
Name: count, dtype: int64

LC profile shape: (4, 25)
Distress levels in profiles: ['high', 'low', 'moderate', 'severe']


---
All 4 profiles confirmed, full 2.26M rows contributing. The "moderate" and "high" buckets are smaller but still have thousands of rows each — more than enough for reliable aggregate stats.

---

In [6]:
# Merge LC aggregate profiles onto UCI customers by distress level
uci_merged = uci.merge(lc_profiles, on='distress_level', how='left')

# Add realistic noise so not every "low" customer has identical LC stats
# Without noise, all 10,253 "low" customers would have the exact same values
np.random.seed(42)
lc_feature_cols = [c for c in uci_merged.columns if c.startswith('lc_') and c.endswith('_mean')]

for col in lc_feature_cols:
    std_col = col.replace('_mean', '_std')
    if std_col in uci_merged.columns:
        noise = np.random.normal(0, 1, len(uci_merged)) * uci_merged[std_col] * 0.3
        uci_merged[col] = uci_merged[col] + noise
        # Floor at 0 for columns that can't be negative
        if any(keyword in col for keyword in ['amnt', 'inc', 'acc', 'installment']):
            uci_merged[col] = uci_merged[col].clip(lower=0)

# Drop the std columns — they were only needed for noise generation
std_cols = [c for c in uci_merged.columns if c.startswith('lc_') and c.endswith('_std')]
uci_merged.drop(columns=std_cols, inplace=True)

print(f"Shape after LC merge: {uci_merged.shape}")
print(f"New LC columns: {[c for c in uci_merged.columns if c.startswith('lc_')]}")
print(f"Null check: {uci_merged[[c for c in uci_merged.columns if c.startswith('lc_')]].isnull().sum().sum()}")

Shape after LC merge: (30000, 47)
New LC columns: ['lc_loan_amnt_mean', 'lc_funded_amnt_mean', 'lc_annual_inc_mean', 'lc_dti_mean', 'lc_revol_util_mean', 'lc_delinq_2yrs_mean', 'lc_inq_last_6mths_mean', 'lc_open_acc_mean', 'lc_pub_rec_mean', 'lc_total_acc_mean', 'lc_installment_mean', 'lc_int_rate_mean']
Null check: 0


## Step 3: Assign Sparkov Spending Profiles
Sparkov has 999 customers with full transaction histories. We compute 
monthly spending profiles per customer (total spend, volatility, top category), 
then assign each Sparkov profile to ~30 UCI customers based on similarity.

In [7]:
# Parse transaction dates
sparkov['transaction_date'] = pd.to_datetime(sparkov['transaction_date'])

# Compute per-customer spending profiles from full transaction history
sparkov_profiles = sparkov.groupby('customer_id').agg(
    total_spend=('amount', 'sum'),
    avg_monthly_spend=('amount', 'mean'),
    spend_volatility=('amount', 'std'),
    num_transactions=('amount', 'count'),
    top_category=('category', lambda x: x.mode()[0]),
    fraud_rate=('is_fraud', 'mean')
).reset_index()

# Rank customers by spending level (quartiles)
sparkov_profiles['spend_quartile'] = pd.qcut(
    sparkov_profiles['total_spend'], 
    q=4, labels=['q1_low', 'q2_mid', 'q3_high', 'q4_heavy']
)

print(f"Sparkov profiles: {sparkov_profiles.shape}")
print(f"\n=== Spend Quartile Distribution ===")
print(sparkov_profiles['spend_quartile'].value_counts().sort_index())
print(f"\n=== Top 5 Categories ===")
print(sparkov_profiles['top_category'].value_counts().head())

Sparkov profiles: (999, 8)

=== Spend Quartile Distribution ===
spend_quartile
q1_low      250
q2_mid      250
q3_high     249
q4_heavy    250
Name: count, dtype: int64

=== Top 5 Categories ===
top_category
gas_transport    475
grocery_pos      179
shopping_pos     136
home             118
shopping_net      40
Name: count, dtype: int64


---
999 profiles, even quartile split. Now we map them onto UCI.

---

In [8]:
# Strategy: Match UCI customers to Sparkov profiles by financial behavior
# - Low distress → q1/q2 spenders (normal spending)
# - Moderate → q2/q3 (slightly elevated)
# - High → q3/q4 (high spending, possible stress spending)
# - Severe → q4 (heavy spending or erratic)

distress_to_quartile = {
    'low': ['q1_low', 'q2_mid'],
    'moderate': ['q2_mid', 'q3_high'],
    'high': ['q3_high', 'q4_heavy'],
    'severe': ['q4_heavy']
}

np.random.seed(42)

# For each UCI customer, randomly pick a Sparkov profile from matching quartiles
assigned_profiles = []

for _, row in uci_merged.iterrows():
    distress = row['distress_level']
    eligible_quartiles = distress_to_quartile[distress]
    eligible = sparkov_profiles[sparkov_profiles['spend_quartile'].isin(eligible_quartiles)]
    chosen = eligible.sample(1, random_state=np.random.randint(0, 100000))
    assigned_profiles.append(chosen.iloc[0])

sparkov_assigned = pd.DataFrame(assigned_profiles).reset_index(drop=True)

# Rename to avoid column conflicts
sparkov_cols = {
    'total_spend': 'sp_total_spend',
    'avg_monthly_spend': 'sp_avg_monthly_spend',
    'spend_volatility': 'sp_spend_volatility',
    'num_transactions': 'sp_num_transactions',
    'top_category': 'sp_top_category',
    'fraud_rate': 'sp_fraud_rate'
}
sparkov_assigned = sparkov_assigned.rename(columns=sparkov_cols)

# Drop Sparkov customer_id and quartile — not needed in final
sparkov_assigned = sparkov_assigned.drop(columns=['customer_id', 'spend_quartile'])

# Concat side by side
uci_merged = pd.concat([uci_merged, sparkov_assigned], axis=1)

print(f"Shape after Sparkov merge: {uci_merged.shape}")
print(f"New Sparkov columns: {list(sparkov_cols.values())}")
print(f"Null check: {uci_merged[list(sparkov_cols.values())].isnull().sum().sum()}")

Shape after Sparkov merge: (30000, 53)
New Sparkov columns: ['sp_total_spend', 'sp_avg_monthly_spend', 'sp_spend_volatility', 'sp_num_transactions', 'sp_top_category', 'sp_fraud_rate']
Null check: 0


---
30K × 53, clean. Now the NLP merges — these are the most important ones for our FinBERT branch.

---

## Step 4: Assign Financial Sentiment Text
PhraseBank sentences get mapped by distress level:
- Low/moderate → neutral/positive text (stable financial language)
- High → mixed negative/neutral
- Severe → negative text (financial distress language)

In [9]:
# Reload PhraseBank
phrasebank = pd.read_csv("../data/processed/financial_phrasebank_processed.csv")

print("=== PhraseBank Label Distribution ===")
print(phrasebank['label'].value_counts())

# Map distress levels to eligible sentiment labels
distress_to_sentiment = {
    'low': ['positive', 'neutral'],
    'moderate': ['neutral'],
    'high': ['neutral', 'negative'],
    'severe': ['negative']
}

np.random.seed(42)

# Assign a PhraseBank sentence to each UCI customer
assigned_sentences = []

for _, row in uci_merged.iterrows():
    distress = row['distress_level']
    eligible_labels = distress_to_sentiment[distress]
    pool = phrasebank[phrasebank['label'].isin(eligible_labels)]
    chosen = pool.sample(1, random_state=np.random.randint(0, 100000))
    assigned_sentences.append({
        'pb_sentence': chosen.iloc[0]['sentence'],
        'pb_label': chosen.iloc[0]['label']
    })

pb_assigned = pd.DataFrame(assigned_sentences)
uci_merged = pd.concat([uci_merged, pb_assigned], axis=1)

print(f"\nShape after PhraseBank merge: {uci_merged.shape}")
print(f"\n=== Assigned PhraseBank Labels ===")
print(uci_merged['pb_label'].value_counts())

=== PhraseBank Label Distribution ===
label
neutral     1391
positive     570
negative     303
Name: count, dtype: int64

Shape after PhraseBank merge: (30000, 55)

=== Assigned PhraseBank Labels ===
pb_label
neutral     23693
negative     3380
positive     2927
Name: count, dtype: int64


## Step 5: Assign Mental Health Sentiment Text
Mental Health statements add emotional language dimension.
High/severe distress customers get anxiety/stress text.
Low/moderate get calmer statements.

In [10]:
# Reload Mental Health
mental = pd.read_csv("../data/processed/mental_health_processed.csv")

print("=== Mental Health Status Distribution ===")
print(mental['status'].value_counts())

=== Mental Health Status Distribution ===
status
Normal        16343
Depression    15404
Anxiety        3841
Stress         2587
Name: count, dtype: int64


In [11]:
# Map distress levels to mental health statuses
distress_to_mental = {
    'low': ['Normal'],
    'moderate': ['Normal', 'Stress'],
    'high': ['Stress', 'Anxiety'],
    'severe': ['Anxiety', 'Depression']
}

np.random.seed(42)

assigned_mental = []

for _, row in uci_merged.iterrows():
    distress = row['distress_level']
    eligible_statuses = distress_to_mental[distress]
    pool = mental[mental['status'].isin(eligible_statuses)]
    chosen = pool.sample(1, random_state=np.random.randint(0, 100000))
    assigned_mental.append({
        'mh_statement': chosen.iloc[0]['statement'],
        'mh_status': chosen.iloc[0]['status']
    })

mh_assigned = pd.DataFrame(assigned_mental)
uci_merged = pd.concat([uci_merged, mh_assigned], axis=1)

print(f"Shape after Mental Health merge: {uci_merged.shape}")
print(f"\n=== Assigned Mental Health Status ===")
print(uci_merged['mh_status'].value_counts())

Shape after Mental Health merge: (30000, 57)

=== Assigned Mental Health Status ===
mh_status
Normal        21147
Stress         3487
Anxiety        3282
Depression     2084
Name: count, dtype: int64


## Step 6: Final Review & Save
All 5 datasets merged. Quick sanity check before saving.

In [12]:
# Final overview
print(f"=== FINAL MERGED DATASET ===")
print(f"Shape: {uci_merged.shape}")
print(f"\n--- Column Groups ---")

col_groups = {
    'UCI Original': [c for c in uci_merged.columns if not c.startswith(('lc_', 'sp_', 'pb_', 'mh_', 'customer_id', 'avg_pay_delay', 'distress_level'))],
    'Engineered': ['customer_id', 'avg_pay_delay', 'distress_level'],
    'Lending Club': [c for c in uci_merged.columns if c.startswith('lc_')],
    'Sparkov': [c for c in uci_merged.columns if c.startswith('sp_')],
    'PhraseBank': [c for c in uci_merged.columns if c.startswith('pb_')],
    'Mental Health': [c for c in uci_merged.columns if c.startswith('mh_')]
}

for group, cols in col_groups.items():
    print(f"\n{group} ({len(cols)} cols): {cols}")

# Null check across everything
total_nulls = uci_merged.isnull().sum().sum()
print(f"\n=== Total nulls in entire dataset: {total_nulls} ===")

# Save
output_path = "../data/processed/sbdr_merged_dataset.csv"
uci_merged.to_csv(output_path, index=False)
print(f"\nSaved to {output_path}")
print(f"File size: {pd.io.common.file_exists(output_path)}")

=== FINAL MERGED DATASET ===
Shape: (30000, 57)

--- Column Groups ---

UCI Original (32 cols): ['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'default payment next month', 'spending_volatility', 'pay_ratio_1', 'pay_ratio_2', 'pay_ratio_3', 'pay_ratio_4', 'pay_ratio_5', 'pay_ratio_6', 'delinq_accel']

Engineered (3 cols): ['customer_id', 'avg_pay_delay', 'distress_level']

Lending Club (12 cols): ['lc_loan_amnt_mean', 'lc_funded_amnt_mean', 'lc_annual_inc_mean', 'lc_dti_mean', 'lc_revol_util_mean', 'lc_delinq_2yrs_mean', 'lc_inq_last_6mths_mean', 'lc_open_acc_mean', 'lc_pub_rec_mean', 'lc_total_acc_mean', 'lc_installment_mean', 'lc_int_rate_mean']

Sparkov (6 cols): ['sp_total_spend', 'sp_avg_monthly_spend', 'sp_spend_volatility', 'sp_num_transactions', 'sp_top_category', 'sp_fra

In [14]:
# Quick peek at a sample customer
print("=== Sample Customer Profile ===")
sample = uci_merged[uci_merged['distress_level'] == 'severe'].iloc[0]
print(f"\nCustomer: {sample['customer_id']}")
print(f"Distress Level: {sample['distress_level']}")
print(f"Default: {sample['default payment next month']}")
print(f"Avg Pay Delay: {sample['avg_pay_delay']:.2f}")
print(f"\nLC Profile: loan_amnt={sample['lc_loan_amnt_mean']:.0f}, dti={sample['lc_dti_mean']:.1f}, int_rate={sample['lc_int_rate_mean']:.1f}%")
print(f"Sparkov: total_spend={sample['sp_total_spend']:.0f}, volatility={sample['sp_spend_volatility']:.1f}, top_cat={sample['sp_top_category']}")
print(f"\nPhraseBank: [{sample['pb_label']}] {sample['pb_sentence'][:100]}...")
print(f"Mental Health: [{sample['mh_status']}] {sample['mh_statement'][:100]}...")

=== Sample Customer Profile ===

Customer: SBDR_00013
Distress Level: severe
Default: 1
Avg Pay Delay: 1.17

LC Profile: loan_amnt=10505, dti=19.3, int_rate=17.0%
Sparkov: total_spend=212337, volatility=127.0, top_cat=gas_transport

PhraseBank: [negative] Finnair believes the strike will cause it daily net losses in excess of EUR 2mn due to canceled rese...
Mental Health: [Depression] No one cares. No one is interested. My life is a joke. I feel like I have no one. What is the point?...


---
That's a complete multi-modal customer right there. SBDR_00013 tells a coherent story — severe distress, actually defaulted, high DTI, high interest rate, negative financial sentiment, depression-level emotional text. Every signal points the same direction. This is exactly what the XGBoost model will learn to read across all three branches.

---

## ✅ Notebook 03 Complete — Phase A, Step A3

**What we did:**
- Assigned 30K unique customer IDs to UCI anchor dataset
- Classified customers into 4 distress levels from payment history
- Merged Lending Club aggregate profiles (12 features) with noise for realism
- Assigned Sparkov spending profiles (6 features) matched by distress-spending similarity
- Mapped PhraseBank financial sentiment text by distress → sentiment alignment
- Mapped Mental Health emotional text by distress → psychological state alignment

**Final dataset:** `data/processed/sbdr_merged_dataset.csv`
- **30,000 customers × 57 columns**
- Zero nulls
- Each customer has: payment sequences + demographics + loan metrics + spending profile + financial text + emotional text

**Distress distribution:**
- Low: 10,253 | Moderate: 12,614 | High: 4,500 | Severe: 2,633

**Next up → Step A4:** Full feature engineering on merged dataset